<a href="https://colab.research.google.com/github/Jones-Rozario/gen-ai/blob/main/GPT_2Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets torch sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:

# Step 2: Expanded Custom Dataset with Multi-line Dialogues
custom_dataset = """
SCENE 1: INT. COFFEE SHOP - DAY
Alice drums her fingers on the table, glancing at her watch.
ALICE: "He's 20 minutes late."
BARISTA: "Another coffee while you wait?"
ALICE: "No thanks, I've had enough caffeine for today."

SCENE 2: EXT. PARK - NIGHT
A shadowy figure approaches a bench where a package sits.
FIGURE: "The drop was supposed to be at midnight."
SECOND FIGURE (from shadows): "Plans changed."
FIGURE: "This isn't how we agreed to do things."

SCENE 3: INT. LAB - LATE NIGHT
Scientists cluster around a glowing device that pulses erratically.
DR. LEE: "The quantum readings are off the charts!"
TECHNICIAN: "Should we shut it down?"
INTERN: "But think of what we could learn!"

REVIEW: "This blender is a powerhouse!"
REVIEW: "Stopped working after 3 uses."
REVIEW: "Customer service replaced it immediately."
REVIEW: "Worth every penny for the smoothies alone."
"""

# Save to file
with open("expanded_data.txt", "w") as f:
    f.write(custom_dataset.strip())

# Step 3: Load and Tokenize Data with Proper Formatting
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Essential for padding

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,  # Increased for longer dialogues
        padding="max_length"
    )

dataset = load_dataset("text", data_files={"train": "expanded_data.txt"})
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Critical: Add labels for language modeling
tokenized_dataset = tokenized_dataset.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

# Step 4: Configure Training with Larger Dataset
from transformers import TrainingArguments, Trainer

model = GPT2LMHeadModel.from_pretrained("gpt2")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Must be False for GPT-2
)

training_args = TrainingArguments(
    output_dir="./gpt2-dialogue-trained",
    num_train_epochs=5,  # Increased for better learning
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)

# Step 5: Run Training
trainer.train()

# Step 6: Generate Sample Dialogues
from transformers import pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

print("MOVIE DIALOGUE GENERATION:")
print(generator(
    "SCENE 4: INT. SPACESHIP BRIDGE\nCAPTAIN: \"Status report!\"\n",
    max_length=150,
    num_return_sequences=1
)[0]['generated_text'])

print("\nPRODUCT REVIEW GENERATION:")
print(generator(
    "REVIEW: \"This gaming laptop\"",
    max_length=100,
    num_return_sequences=1
)[0]['generated_text'])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Step,Training Loss


Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


MOVIE DIALOGUE GENERATION:
SCENE 4: INT. SPACESHIP BRIDGE
CAPTAIN: "Status report!"
JASON 1: "Status report received."

JACKSISTER: "I'm sorry, but this is not the end."

JACKY: "It's okay, it's not."
JACKY: "Thanks, Captain."

JACKY: "Sorry."

(pause) "What happened?"

JACKY: "The ship moved to get it."

JACKY: "It's okay."

JACKY: "We're getting things where we shouldn't."

JACKY: "But we should get it's working."


PRODUCT REVIEW GENERATION:
REVIEW: "This gaming laptop"

"This laptop is stunning!"

"Extremely recommendible!"

OVERALL: "Perfect for anyone looking to put their money off gaming"


Prompt Engineering **Experiment**

In [17]:
prompt = input("Enter your prompt: ")
print(generator(
    prompt,
    max_length=100,
    num_return_sequences=1
)[0]['generated_text'])

Enter your prompt: Review: this samsung phone
Review: this samsung phone isn't exactly what you want it to be...but it's still the best Samsung Galaxy phone you could buy on the cheap. It's a great deal, and it's also a great deal for the price.
